In [1]:
import random, math, numpy as np
random.seed(1); np.random.seed(1)
CO2_PER_WH=0.45; WATER_PER_WH=1.8; VAN_CO2_PER_KM=250
def footprint(wh): return wh*CO2_PER_WH, wh*WATER_PER_WH

In [2]:
class CrowdReflexAgent:
    def __init__(self, eco=True): self.eco=eco; self.wh=0; self.alerts=0
    def act(self, speed):
        abnormal = speed>3.0 or speed<0.05
        if self.eco:
            self.wh += 0.5/60
            if abnormal: self.wh += 5/60; self.alerts+=1; return 'ALERT'
            return 'idle'
        self.wh += 5/60
        if abnormal: self.alerts+=1; return 'ALERT'
        return 'idle'
speeds=np.concatenate([np.random.normal(1.2,0.3,550),np.random.normal(4,0.5,10)]); np.random.shuffle(speeds)
for eco in (False,True):
    a=CrowdReflexAgent(eco); [a.act(s) for s in speeds]
    print('eco' if eco else 'normal', 'alerts:',a.alerts,'energy Wh: %.2f'%a.wh,'CO2 g: %.2f'%footprint(a.wh)[0],'water ml: %.2f'%footprint(a.wh)[1])

normal alerts: 10 energy Wh: 46.67 CO2 g: 21.00 water ml: 84.00
eco alerts: 10 energy Wh: 5.50 CO2 g: 2.48 water ml: 9.90


In [3]:
ACTIONS={
 'do_nothing':(0,0),'dim_lights':(2,1),'play_calm_music':(3,2),'guided_breathing_app':(4,3),
 'aircon_cooling':(3,40),'call_nurse':(8,6)}
def utility(a,stress,w_eco=0.08):
    red,wh=ACTIONS[a]; return min(red,stress)-w_eco*wh
def choose(stress,eco=True):
    if not eco: return max(ACTIONS,key=lambda a:ACTIONS[a][0])
    return max(ACTIONS,key=lambda a:utility(a,stress))
tot={True:0,False:0}
for s in [2,4,6,9,3,7,8,1]:
    for eco in (True,False):
        a=choose(s,eco); tot[eco]+=ACTIONS[a][1]
    print('stress',s,'| eco picks',choose(s,True),'| normal picks',choose(s,False))
for eco in (True,False): print('eco' if eco else 'normal','Wh:',tot[eco],'CO2 g: %.1f'%footprint(tot[eco])[0])

stress 2 | eco picks dim_lights | normal picks call_nurse
stress 4 | eco picks guided_breathing_app | normal picks call_nurse
stress 6 | eco picks call_nurse | normal picks call_nurse
stress 9 | eco picks call_nurse | normal picks call_nurse
stress 3 | eco picks play_calm_music | normal picks call_nurse
stress 7 | eco picks call_nurse | normal picks call_nurse
stress 8 | eco picks call_nurse | normal picks call_nurse
stress 1 | eco picks dim_lights | normal picks call_nurse
eco Wh: 31 CO2 g: 14.0
normal Wh: 48 CO2 g: 21.6


In [4]:
import itertools
pts={'School':(0,0),'S1':(2,1),'S2':(3,4),'S3':(-2,3),'S4':(-3,-2)}
def d(a,b): return math.dist(pts[a],pts[b])
unsafe={('S1','S4'),('S4','S1')}
def route_len(order):
    r=['School']+list(order)+['School']; L=0
    for a,b in zip(r,r[1:]):
        if (a,b) in unsafe: return None
        L+=d(a,b)
    return L
stops=['S1','S2','S3','S4']
valid={o:route_len(o) for o in itertools.permutations(stops) if route_len(o)}
best=min(valid,key=valid.get); naive=('S2','S4','S3','S1')
for name,o in (('naive',naive),('goal-based best',best)):
    L=route_len(o) or float('nan'); print(name,o,'km: %.2f'%L,'CO2 kg: %.2f'%(L*VAN_CO2_PER_KM/1000))
print('Saved CO2 (g):', round((route_len(naive)-valid[best])*VAN_CO2_PER_KM,1) if route_len(naive) else 'naive route is unsafe')

naive ('S2', 'S4', 'S3', 'S1') km: 25.29 CO2 kg: 6.32
goal-based best ('S1', 'S2', 'S3', 'S4') km: 19.20 CO2 kg: 4.80
Saved CO2 (g): 1522.6


In [5]:
acts=['drive_to_gym','cycle_to_park','walk_with_friends','home_workout']
engage={'drive_to_gym':0.55,'cycle_to_park':0.65,'walk_with_friends':0.6,'home_workout':0.5}
co2={'drive_to_gym':800,'cycle_to_park':0,'walk_with_friends':0,'home_workout':5}
Qv={a:0 for a in acts}; N={a:0 for a in acts}; hist=[]; tot_co2=0
for t in range(500):
    a=random.choice(acts) if random.random()<0.1 else max(Qv,key=Qv.get)
    did=random.random()<engage[a]
    r=(1 if did else 0)-co2[a]/1000
    N[a]+=1; Qv[a]+=(r-Qv[a])/N[a]; tot_co2+=co2[a]*did; hist.append(a)
print({a:round(v,3) for a,v in Qv.items()}); print('chosen counts',N); print('total CO2 g:',tot_co2)
rand_co2=sum(co2[random.choice(acts)]*(random.random()<0.6) for _ in range(500)); print('random agent CO2 g ~',rand_co2)

{'drive_to_gym': -0.527, 'cycle_to_park': 0.537, 'walk_with_friends': 0.607, 'home_workout': 0.495}
chosen counts {'drive_to_gym': 11, 'cycle_to_park': 54, 'walk_with_friends': 425, 'home_workout': 10}
total CO2 g: 2425
random agent CO2 g ~ 61210


## Description and Assumptions

In this question, each agent is implemented in Python and its environmental impact is estimated from the energy it uses. Since the question does not provide conversion values, the following assumptions are used throughout the notebook:

- **Electricity to CO₂:** 0.45 g CO₂ per Wh
- **Electricity to water:** 1.8 ml of water per Wh, representing water used for cooling during power generation
- **Diesel van:** 250 g CO₂ per km

The descriptions below explain what each agent is doing and how its decisions affect energy use, CO₂ emissions, water use, or travel emissions.

## A. Reflex Agent — Abnormal Movement in a Crowd

The reflex agent uses the **average speed of the crowd** as its percept. It treats movement as abnormal when the speed is above **3 m/s**, which represents running, or below **0.05 m/s**, which represents a sudden freeze. In either case, the agent raises an alert.

The eco-friendly version is designed to reduce unnecessary camera use. A **0.5 W motion sensor** stays on continuously, while the **5 W camera** is switched on only when abnormal movement is detected. In comparison, the normal agent keeps the camera on continuously.

For the 560 crowd-speed readings used in the simulation, both agents detected the same **10 abnormal events**, so the detection performance was unchanged. However, the normal agent consumed **46.67 Wh**, while the eco-friendly agent used only **5.50 Wh**.

Using the assumed conversion rates, this reduced CO₂ emissions from **21.00 g to 2.48 g** and water use from **84.00 ml to 9.90 ml**. This shows that the eco agent can provide the same alerts while using much less energy.

## B. Utility-Based Agent — Stressed and Tense Patients

The utility-based agent chooses an action according to both the expected stress reduction and the energy required. Its possible actions are **doing nothing, dimming the lights, playing calm music, using a guided breathing app, using air conditioning, or calling a nurse**.

For the eco-friendly agent, the utility is calculated as:

**Utility = stress reduction − 0.08 × energy use**

The agent then selects the action with the highest utility for the patient's current stress level. The normal agent, on the other hand, simply chooses the action with the largest stress-relief value, without considering energy use.

For the 8 patients in the simulation, the eco agent used lower-energy options for mild stress, such as dimming the lights, calm music, or the breathing app. When the stress level was **6 or higher**, it still selected **calling the nurse**, so the energy-saving approach did not replace the required higher-level care.

Overall, the eco agent used **31 Wh**, compared with **48 Wh** for the normal agent. This corresponds to **14.0 g of CO₂ instead of 21.6 g**, showing that utility-based decision-making can reduce energy use while still responding appropriately to high-stress cases.

## C. Goal-Based Agent — School Van Network

The goal of the school van agent is to **pick up all four students and return to school using safe roads while keeping the total distance as short as possible**.

The agent checks all possible orders of the four student stops. It removes any route that contains the unsafe connection between **S1 and S4**, and then selects the shortest remaining safe route.

The normal comparison route is **School → S2 → S4 → S3 → S1 → School**, which covers **25.29 km** and produces approximately **6.32 kg of CO₂** using the assumed diesel-van emission rate.

The goal-based agent selects **School → S1 → S2 → S3 → S4 → School**, with a total distance of **19.20 km** and approximately **4.80 kg of CO₂**.

Therefore, the selected route saves about **6.09 km** and **1.52 kg of CO₂ per trip** compared with the normal route, while also respecting the safety constraint.

## D. Learning Agent — Sports Activities for Young People

The learning agent recommends one of four activities: **driving to the gym, cycling to the park, walking with friends, or doing a home workout**. It uses an **epsilon-greedy strategy with ε = 0.1**, meaning it usually chooses the activity it currently believes is best but sometimes explores another activity.

The reward is based on whether the person actually does the suggested activity, with a penalty for the CO₂ produced by car travel. This encourages the agent to learn activities that are both engaging and environmentally friendly.

After **500 simulated days**, the agent learned to prefer **walking with friends**, which was selected **425 times**. The car-based gym trip was selected only **11 times**, while cycling was selected **54 times** and home workouts **10 times**.

The total CO₂ produced by the learning agent was **2,425 g**. In comparison, the random-action baseline in this run produced approximately **61,210 g of CO₂**. This shows how learning from previous outcomes can move the agent toward activities that are more sustainable while still encouraging participation.